# 01 - Exploratory Data Analysis: Spatiotemporal Patterns

This notebook performs comprehensive exploratory data analysis on Manhattan motor vehicle collision data, examining temporal patterns (hourly, daily, monthly) and spatial distributions (precinct-level, CBD vs non-CBD).

**Runtime:** ~5 minutes  
**Data Required:** `crashes_manhattan.csv` (processed)

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally")
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

## Load Crash Data

In [ ]:
crashes = pd.read_csv(os.path.join(PROCESSED_DIR, 'crashes_manhattan.csv'),
                         parse_dates=['crash_datetime'])
print(f"Total crashes: {len(crashes):,}")
print(f"Date range: {crashes['crash_datetime'].min()} to {crashes['crash_datetime'].max()}")
print(f"Duration: {(crashes['crash_datetime'].max() - crashes['crash_datetime'].min()).days:,} days")
print(f"\nColumns: {list(crashes.columns)}")
crashes.info(memory_usage='deep')

## Dataset Overview Statistics

In [ ]:
total = len(crashes)
date_range = (crashes['crash_datetime'].max() - crashes['crash_datetime'].min()).days
avg_per_day = total / date_range
avg_per_hour = avg_per_day / 24

print("=" * 60)
print("KEY STATISTICS - Manhattan Crash Demand")
print("=" * 60)
print(f"  Total crashes: {total:,}")
print(f"  Date range: {crashes['crash_datetime'].min().date()} to {crashes['crash_datetime'].max().date()}")
print(f"  Duration: {date_range:,} days ({date_range/365.25:.1f} years)")
print(f"  Average crashes per day: {avg_per_day:.1f}")
print(f"  Average crashes per hour: {avg_per_hour:.2f}")

if 'in_cbd' in crashes.columns:
    cbd = crashes['in_cbd'].sum()
    non_cbd = total - cbd
    print(f"\n  CBD crashes: {cbd:,} ({100*cbd/total:.1f}%)")
    print(f"  Non-CBD crashes: {non_cbd:,} ({100*non_cbd/total:.1f}%)")

## Temporal Analysis
### Hourly Distribution

In [ ]:
crashes['hour'] = crashes['crash_datetime'].dt.hour
hourly = crashes.groupby('hour').size()

fig, ax = plt.subplots(figsize=(14, 6))
hourly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Total Crashes')
ax.set_title('Crash Distribution by Hour of Day (Manhattan)')
ax.axhline(y=hourly.mean(), color='red', linestyle='--', label=f'Mean: {hourly.mean():,.0f}')
ax.legend()
plt.tight_layout()
save_output(fig, 'hourly_distribution.png', 'figures/eda')
plt.show()

peak_hour = hourly.idxmax()
print(f"Peak hour: {peak_hour}:00 ({hourly.max():,} total crashes)")
print(f"Quietest hour: {hourly.idxmin()}:00 ({hourly.min():,} total crashes)")

### Day-of-Week Distribution

In [ ]:
crashes['dow'] = crashes['crash_datetime'].dt.dayofweek
crashes['day_name'] = crashes['crash_datetime'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = crashes.groupby('day_name').size().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 6))
daily.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Total Crashes')
ax.set_title('Crash Distribution by Day of Week (Manhattan)')
ax.axhline(y=daily.mean(), color='red', linestyle='--', label=f'Mean: {daily.mean():,.0f}')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_output(fig, 'dow_distribution.png', 'figures/eda')
plt.show()

print(f"Peak day: {daily.idxmax()} ({daily.max():,})")

### Monthly Distribution

In [ ]:
crashes['month'] = crashes['crash_datetime'].dt.month
monthly = crashes.groupby('month').size()

fig, ax = plt.subplots(figsize=(10, 6))
monthly.plot(kind='bar', ax=ax, color='seagreen', edgecolor='white')
ax.set_xlabel('Month')
ax.set_ylabel('Total Crashes')
ax.set_title('Crash Distribution by Month (Manhattan)')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
plt.tight_layout()
save_output(fig, 'monthly_distribution.png', 'figures/eda')
plt.show()

### Hourly Heatmap by Day of Week

In [ ]:
heatmap_data = crashes.groupby(['dow', 'hour']).size().unstack(fill_value=0)
heatmap_data.index = dow_order

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='d', ax=ax)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
ax.set_title('Crash Frequency Heatmap: Day-of-Week x Hour-of-Day')
plt.tight_layout()
save_output(fig, 'dow_hour_heatmap.png', 'figures/eda')
plt.show()

## Spatial Analysis
### Crashes by Precinct

In [ ]:
if 'LATITUDE' in crashes.columns and 'LONGITUDE' in crashes.columns:
    valid_coords = crashes.dropna(subset=['LATITUDE', 'LONGITUDE'])
    print(f"Crashes with valid coordinates: {len(valid_coords):,} ({100*len(valid_coords)/len(crashes):.1f}%)")

    fig, ax = plt.subplots(figsize=(10, 12))
    ax.scatter(valid_coords['LONGITUDE'].values[::10],
               valid_coords['LATITUDE'].values[::10],
               alpha=0.05, s=1, c='red')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Manhattan Crash Locations (every 10th point)')
    ax.set_aspect('equal')
    plt.tight_layout()
    save_output(fig, 'crash_scatter.png', 'figures/eda')
    plt.show()

### CBD vs Non-CBD Comparison

In [ ]:
if 'in_cbd' in crashes.columns:
    cbd_crashes = crashes[crashes['in_cbd'] == True]
    non_cbd_crashes = crashes[crashes['in_cbd'] == False]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Hourly comparison
    cbd_hourly = cbd_crashes.groupby('hour').size()
    non_cbd_hourly = non_cbd_crashes.groupby('hour').size()

    axes[0].plot(cbd_hourly.index, cbd_hourly.values, 'b-o', label=f'CBD ({len(cbd_crashes):,})', markersize=4)
    axes[0].plot(non_cbd_hourly.index, non_cbd_hourly.values, 'r-s', label=f'Non-CBD ({len(non_cbd_crashes):,})', markersize=4)
    axes[0].set_xlabel('Hour of Day')
    axes[0].set_ylabel('Total Crashes')
    axes[0].set_title('Hourly Pattern: CBD vs Non-CBD')
    axes[0].legend()

    # Day of week comparison
    cbd_dow = cbd_crashes.groupby('day_name').size().reindex(dow_order)
    non_cbd_dow = non_cbd_crashes.groupby('day_name').size().reindex(dow_order)
    x = range(7)
    w = 0.35
    axes[1].bar([i - w/2 for i in x], cbd_dow.values, w, label='CBD', color='steelblue')
    axes[1].bar([i + w/2 for i in x], non_cbd_dow.values, w, label='Non-CBD', color='coral')
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(dow_order, rotation=45)
    axes[1].set_ylabel('Total Crashes')
    axes[1].set_title('Day-of-Week: CBD vs Non-CBD')
    axes[1].legend()

    plt.tight_layout()
    save_output(fig, 'cbd_comparison.png', 'figures/eda')
    plt.show()

### Time Series Trend

In [ ]:
crashes['year_month'] = crashes['crash_datetime'].dt.to_period('M')
monthly_ts = crashes.groupby('year_month').size()

fig, ax = plt.subplots(figsize=(16, 6))
monthly_ts.plot(ax=ax, color='steelblue')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Crashes per Month')
ax.set_title('Monthly Crash Volume Over Time (Manhattan)')

# Add rolling average
rolling = monthly_ts.rolling(12).mean()
ax.plot(rolling.index.astype(str), rolling.values, 'r-', linewidth=2, label='12-month rolling avg')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_output(fig, 'monthly_trend.png', 'figures/eda')
plt.show()

## Summary

Key findings from EDA:
- Manhattan crash data spans 13+ years with 416K+ records
- Clear hourly pattern: peaks at 16:00, lowest at 04:00-05:00
- Friday is the peak day; weekend volumes are lower
- CBD accounts for ~56% of total crashes
- Clear seasonal patterns with summer peaks